In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import re
from sklearn.naive_bayes import MultinomialNB

# ----- VERİ ÖN İŞLEME FONKSİYONLARI (FAZ 2'DEN) -----

def turkish_lower(text):
    text = text.replace('I', 'ı')
    text = text.replace('İ', 'i')
    return text.lower()

def temel_temizleme(text):
    if not isinstance(text, str):
        return ""
    text = turkish_lower(text)
    text = re.sub(r'[^a-z0-9ıöüçğş\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Kısaltmalar için basit bir sözlük
standardizasyon_sozlugu = {
    r'\bmah\b': 'mahallesi', r'\bmh\b': 'mahallesi',
    r'\bsok\b': 'sokak', r'\bsk\b': 'sokak',
    r'\bcad\b': 'caddesi', r'\bcd\b': 'caddesi',
    r'\bno\b': 'numara', r'\bapt\b': 'apartmani',
    r'\bd\b': 'daire'
}

def adresi_standardize_et(text):
    for kural, degisim in standardizasyon_sozlugu.items():
        text = re.sub(kural, degisim, text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def on_isle(text_series):
    print("Temel temizleme...")
    processed = text_series.apply(temel_temizleme)
    print("Standardizasyon...")
    processed = processed.apply(adresi_standardize_et)
    print("Ön işleme tamamlandı.")
    return processed



print("Veri yükleniyor...")
try:
    df = pd.read_csv('train_processed.csv').dropna()
except FileNotFoundError:
    print("train.csv bulunamadı. Lütfen dosya yolunu kontrol edin.")
    exit()

# Adresleri önceden işle
df['processed_address'] = on_isle(df['address'])

# Veriyi train ve validation olarak ayır
X = df['processed_address']
y = df['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Eğitim seti boyutu: {len(X_train)}, Doğrulama seti boyutu: {len(X_val)}")

Veri yükleniyor...
Temel temizleme...
Standardizasyon...
Ön işleme tamamlandı.
Eğitim seti boyutu: 542868, Doğrulama seti boyutu: 135718


In [4]:
# 1. TF-IDF vektörleştiriciyi oluştur
# Bellek sorunlarından kaçınmak için max_features'ı makul bir seviyede tutalım.
vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 6),  # Biraz daha geniş bir aralık deneyelim
    max_features=40000   # Bu değerle başla, gerekirse azaltıp artırabilirsin
)

# 2. Vektörleştiriciyi SADECE EĞİTİM VERİSİNE (X_train) uydur (fit) ve dönüştür (transform)
print("\nTF-IDF vektörleştirici eğitim verisine uygulanıyor...")
X_train_tfidf = vectorizer.fit_transform(X_train)
print("Eğitim verisi vektörleştirildi.")
print(f"Oluşturulan matrisin boyutu: {X_train_tfidf.shape}")

# 3. Aynı vektörleştiriciyi SADECE DOĞRULAMA VERİSİNE (X_val) uygula (sadece transform)
# Burada ASLA .fit_transform() KULLANMA! Bu veri sızıntısı olur.
print("\nTF-IDF vektörleştirici doğrulama verisine uygulanıyor...")
X_val_tfidf = vectorizer.transform(X_val)
print("Doğrulama verisi vektörleştirildi.")
print(f"Oluşturulan matrisin boyutu: {X_val_tfidf.shape}")


TF-IDF vektörleştirici eğitim verisine uygulanıyor...
Eğitim verisi vektörleştirildi.
Oluşturulan matrisin boyutu: (542868, 40000)

TF-IDF vektörleştirici doğrulama verisine uygulanıyor...
Doğrulama verisi vektörleştirildi.
Oluşturulan matrisin boyutu: (135718, 40000)


In [5]:
# 1. Modeli oluştur
# alpha, bir düzgünleştirme (smoothing) parametresidir. 0'a yakın değerler modeli daha karmaşık hale getirir.
# Genellikle 0.01 ile 1.0 arasında bir değer iyi çalışır.
model = MultinomialNB(alpha=0.1)

# 2. Modeli TF-IDF ile dönüştürülmüş eğitim verisiyle eğit
print("\nMultinomial Naive Bayes modeli eğitiliyor...")
model.fit(X_train_tfidf, y_train)
print("Model eğitimi tamamlandı.")

# 3. Doğrulama seti üzerinde tahmin yap
print("\nDoğrulama seti üzerinde tahminler yapılıyor...")
y_pred = model.predict(X_val_tfidf)
print("Tahminler tamamlandı.")

# 4. Performansı değerlendir
macro_f1 = f1_score(y_val, y_pred, average='macro')
weighted_f1 = f1_score(y_val, y_pred, average='weighted')

print(f"\nSONUÇLAR (MultinomialNB):")
print(f"Macro F1 Score: {macro_f1:.4f}")
print(f"Weighted F1 Score: {weighted_f1:.4f}")

# Detaylı rapor
print("\nSınıflandırma Raporu (Örnek):")
# Hata almamak için sadece hem gerçek hem de tahmin edilen etiketlerde bulunanları gösterelim
common_labels = sorted(list(set(y_val) & set(y_pred)))
if len(common_labels) > 20:
    common_labels = common_labels[:20] # Raporu kısa tutmak için

print(classification_report(y_val, y_pred, labels=common_labels))


Multinomial Naive Bayes modeli eğitiliyor...


MemoryError: Unable to allocate 42.0 GiB for an array with shape (542868, 10390) and data type int64